[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RimmaShaf/fmlds-eval-workshop/blob/main/02_tracing_with_phoenix.ipynb) <a href="https://codespaces.new/RimmaShaf/fmlds-eval-workshop?quickstart=1"><img src="https://github.com/codespaces/badge.svg" alt="Open in GitHub Codespaces" width="100"></a> [View on GitHub](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/02_tracing_with_phoenix.ipynb)

# Lab 2: Trace Your Agent

### Instrument the agent with OpenTelemetry and inspect its traces in Phoenix

In notebook 1, you built the agent. Here, OpenInference captures LLM calls automatically,
manual spans make agent and tool steps visible, and every trace goes directly to the shared
Phoenix board on Fly.

The workshop's canonical agent remains in `utils.py`; this notebook attaches a tracer to
that implementation rather than creating a second agent that could drift.


---
## 1. Install the lightweight Phoenix tracing SDKs

These packages let this notebook export OpenTelemetry spans and read them from the shared
Phoenix board. They provide tracing and client access; the workshop server is already running.


In [ ]:
%pip install -q arize-phoenix-otel==0.17.1 arize-phoenix-client==3.7.0 openinference-instrumentation-openai==0.1.76

> Install packages before importing the agent or tracing SDK. If Colab says the
> runtime needs a restart, restart it now and rerun the notebook from the top.


---
## Required action: provide the Phoenix board key

Before running the next code cell, open Colab's **Secrets** panel (key icon), add
`PHOENIX_API_KEY` using the participant board key supplied by the organizer, and enable
Notebook access. This is separate from `LITELLM_API_KEY`. If you do not add the secret,
the bootstrap cell will ask for the key privately.


## 2. Bootstrap the Notebook 1 agent

Notebook 1's agent has been wrapped in the canonical `utils.py` module. This notebook
refreshes that file and imports the same `run_agent` code instead of redefining the agent.
It also loads the ledger into `utils.df`; the local alias `df` below is used only for
ground-truth comparisons.


In [ ]:
import urllib.request
import os
import importlib
from pathlib import Path

utils_path = Path("utils.py")
if not utils_path.exists() or "def set_span_tracer(tracer, attributes=None):" not in utils_path.read_text():
    urllib.request.urlretrieve("https://raw.githubusercontent.com/RimmaShaf/fmlds-eval-workshop/main/utils.py", "utils.py")

import utils
utils = importlib.reload(utils)

utils.load_data()
df = utils.df
utils.connect()
SEAT = utils.seat_tag()
PHOENIX_BOARD_URL = "https://fmlds-eval-workshop-phoenix.fly.dev"
PHOENIX_PROJECT = "00-Part-2-tracing-set-up"
PHOENIX_API_KEY = utils.get_secret("PHOENIX_API_KEY")
if not PHOENIX_API_KEY:
    from getpass import getpass
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
from phoenix.client import Client
phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
try:
    phoenix_client.projects.list()
except Exception as exc:
    if "401" not in str(exc) and "Unauthorized" not in str(exc):
        raise
    from getpass import getpass
    print("Saved Phoenix key was rejected. Enter the participant board key supplied by the organizer.")
    PHOENIX_API_KEY = getpass("Phoenix board API key: ").strip()
    phoenix_client = Client(base_url=PHOENIX_BOARD_URL, api_key=PHOENIX_API_KEY)
    phoenix_client.projects.list()
from utils import NAIVE_SYSTEM_PROMPT, run_agent

print("bootstrap done")
print("Phoenix project:", PHOENIX_PROJECT)
print("Phoenix session.id:", SEAT)


## 3. Send traces to the shared Phoenix board

Register the shared OTLP collector and authenticate with the board key. The whole room
uses the `00-Part-2-tracing-set-up` project, which sorts before the gateway project.


In [ ]:
from phoenix.otel import register

tracer_provider = register(
    project_name=PHOENIX_PROJECT,
    endpoint=f"{PHOENIX_BOARD_URL}/v1/traces",
    protocol="http/protobuf",
    batch=False,
    api_key=PHOENIX_API_KEY,
)
print("OTLP collector:", f"{PHOENIX_BOARD_URL}/v1/traces")


## 4. Why trace the agent?

An answer alone does not show how the agent reached it. OpenInference records each model
call, and the agent adds spans for its router and tools. In Phoenix, follow the trace to see
what the model asked each tool to do, what came back, and where an error occurred.


In [ ]:
from openinference.instrumentation.openai import OpenAIInstrumentor
from openinference.semconv.trace import SpanAttributes

openai_instrumentor = OpenAIInstrumentor()
if openai_instrumentor.is_instrumented_by_opentelemetry:
    openai_instrumentor.uninstrument()
openai_instrumentor.instrument(tracer_provider=tracer_provider)
utils.set_span_tracer(
    tracer_provider.get_tracer(__name__),
    attributes={SpanAttributes.SESSION_ID: SEAT},
)
print("OpenAI calls instrumented; manual agent/chain/tool spans enabled")


> **Trace data note:** spans contain prompts and tool inputs/outputs. This workshop uses
> synthetic transactions; do not send personal or otherwise sensitive data to a tracer.


---
## 5. Challenge 1: signed expenses versus cash flow

These are natural user questions; they do not name a tool. Watch the trace to see what the
router chooses and why.

Ask the agent for October expenses without explaining that income and refunds are positive
while expenses are negative. In the trace, inspect the generated pandas code and tool output.


In [ ]:
gross_october = run_agent(
    "What were my total expenses in October?",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/october-gross-expenses",
)
print(gross_october["answer"])


Compare the answer with two plausible calculations: the positive magnitude of
negative transactions, and the signed total of every October transaction.


In [ ]:
october = df[df["date"].dt.month == 10]
truth_gross_expenses = -october.loc[october["amount"] < 0, "amount"].sum()
signed_cash_flow = october["amount"].sum()
print(f"expense magnitude: {truth_gross_expenses:,.2f}")
print(f"signed cash flow:  {signed_cash_flow:,.2f}")
print(f"positive income/refunds included: {october.loc[october['amount'] > 0, 'amount'].sum():,.2f}")


### Inspect this run's tool I/O

The Phoenix trace shows the overall span tree. This helper also prints the exact tool
arguments and results returned to the agent for this individual run.


In [ ]:
def show_tool_io(result):
    "Print tool arguments and returned results from one agent run."
    for message in result["messages"]:
        if message.get("role") == "assistant":
            for call in message.get("tool_calls") or []:
                function = call["function"]
                print(f"Calling {function['name']}: {function['arguments']}")
        elif message.get("role") == "tool":
            print("Tool result:", message["content"])


In [ ]:
show_tool_io(gross_october)

### Harder sign case: net expenses after refunds

Gross expenses count negative transactions. Net expenses subtract refunds but must exclude
income. The synthetic ledger contains an October refund.


In [ ]:
net_october = run_agent(
    "What were my net expenses in October after refunds, excluding income?",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/net-expenses-after-refunds",
)
print(net_october["answer"])


In [ ]:
refunds = october.loc[
    october["raw_description"].str.contains("REFUND", case=False) & (october["amount"] > 0), "amount"
].sum()
print(f"gross expenses: {truth_gross_expenses:,.2f}")
print(f"refunds:        {refunds:,.2f}")
print(f"net expenses:   {truth_gross_expenses - refunds:,.2f}")


In [ ]:
show_tool_io(net_october)

---
## 6. Challenge 2: merchant aliases and a decoy

Descriptions use both `STARBUCKS` and the alias `SBUX`. Tully's is another coffee shop,
not Starbucks. Inspect the tool input and see whether the generated filter handles both cases.


In [ ]:
starbucks_run = run_agent(
    "How much did I spend at Starbucks in total?",
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label="nb2/starbucks-aliases",
)
print(starbucks_run["answer"])


In [ ]:
is_starbucks = df["raw_description"].str.contains(r"STARBUCKS|SBUX", case=False, regex=True)
truth_starbucks = -df.loc[is_starbucks & (df["amount"] < 0), "amount"].sum()
tully_decoy_count = df["raw_description"].str.contains("TULLY'S", case=False).sum()
print(f"Starbucks + SBUX expense magnitude: {truth_starbucks:,.2f}")
print(f"Tully's decoy rows to exclude:      {tully_decoy_count}")


In [ ]:
show_tool_io(starbucks_run)

---
## 7. Free-text: ask your own question

Try any finance question about the ledger. The prompt below is intentionally open-ended:
the agent chooses its tools, and the trace shows how it responds.


In [ ]:
free_text_query = input("Your question for the financial agent: ")
free_text_result = run_agent(
    free_text_query,
    system_prompt=NAIVE_SYSTEM_PROMPT,
    temperature=0.0,
    label=f"nb2/free-text/{SEAT}",
)
print(free_text_result["answer"])


In [ ]:
show_tool_io(free_text_result)

---
## Find your traces in Phoenix

1. Open the [shared Phoenix board](https://fmlds-eval-workshop-phoenix.fly.dev).
2. Select project **`00-Part-2-tracing-set-up`**.
3. Open the **Traces** table and enter this Phoenix filter expression, replacing
   `your-session-id` with the `session.id` printed during setup:

   `attributes['session.id'] == 'your-session-id'`

4. Open a matching trace and expand its AGENT root to follow the nested router, LLM, tool,
   and tool-generated LLM spans. See [Phoenix filter expressions](https://arize.com/docs/phoenix/tracing/how-to-tracing/filter-expressions).

The code below fetches your own trace spans into `my_spans` so you can inspect them in pandas.


In [ ]:
spans_df = phoenix_client.spans.get_spans_dataframe(project_identifier=PHOENIX_PROJECT, limit=5000)
session_column = "attributes.session.id"
trace_id_column = next((column for column in ("context.trace_id", "trace_id") if column in spans_df.columns), None)
my_trace_ids = []
if session_column in spans_df.columns and trace_id_column:
    my_trace_ids = spans_df.loc[spans_df[session_column] == SEAT, trace_id_column].dropna().unique()
    my_spans = spans_df[spans_df[trace_id_column].isin(my_trace_ids)].copy()
else:
    my_spans = spans_df[spans_df.get(session_column, "") == SEAT].copy() if session_column in spans_df else spans_df.iloc[0:0].copy()
trace_columns = [
    column for column in (
        "name", "span_kind", "parent_id", trace_id_column, session_column,
        "status_code", "attributes.input.value", "attributes.output.value",
        "attributes.llm.token_count.prompt", "attributes.llm.token_count.completion",
    )
    if column and column in my_spans.columns
]
print(f"Loaded {len(my_spans)} spans across {len(my_trace_ids)} of your traces (session.id={SEAT})")
display(my_spans[trace_columns])


---
### Continue the workshop

[Next: Notebook 3 — Evaluations](https://github.com/RimmaShaf/fmlds-eval-workshop/blob/main/03_evaluations.ipynb)
